# Hackology II — Trening na Kaggle

Notebook przygotowany pod środowisko Kaggle (2x T4, read-only input).

**Przed uruchomieniem:**
1. Runtime → Accelerator → GPU T4 x2
2. Dodaj dataset `hackology2-goods` jako Input

## 0. Sprawdź GPU i strukturę danych

In [ ]:
import os
import subprocess

# GPU info
subprocess.run(['nvidia-smi'], check=False)

# Sprawdź strukturę inputu
print('\n=== /kaggle/input ===')
for root, dirs, files in os.walk('/kaggle/input'):
    level = root.replace('/kaggle/input', '').count(os.sep)
    if level > 2:
        continue
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level == 2:
        for f in files[:3]:
            print(f'{indent}  {f}')

## 1. Konfiguracja ścieżek

Zmień `DATASET_NAME` jeśli nazwa folderu w `/kaggle/input` jest inna.

In [ ]:
from pathlib import Path
import os

# Automatycznie znajdź dataset w /kaggle/input
input_root = Path('/kaggle/input')
candidates = list(input_root.rglob('annotations.json'))
if not candidates:
    raise FileNotFoundError('Nie znaleziono annotations.json — sprawdź czy dataset jest dodany')

ANNOTATIONS = candidates[0]
TRAIN_DIR   = ANNOTATIONS.parent          # .../train/
IMAGES_SRC  = TRAIN_DIR / 'images'
INPUT_DIR   = TRAIN_DIR.parent.parent     # root datasetu
TAXONOMY    = next(input_root.rglob('taxonomy.json'))

# Znajdź public_test
test_candidates = list(input_root.rglob('test_images.json'))
TEST_IMAGES_JSON = test_candidates[0] if test_candidates else None
PUBLIC_TEST_IMAGES = TRAIN_DIR.parent / 'public_test' / 'images'

WORK_DIR = Path('/kaggle/working')

print(f'ANNOTATIONS:      {ANNOTATIONS}')
print(f'IMAGES_SRC:       {IMAGES_SRC}')
print(f'TAXONOMY:         {TAXONOMY}')
print(f'TEST_IMAGES_JSON: {TEST_IMAGES_JSON}')
print(f'PUBLIC_TEST:      {PUBLIC_TEST_IMAGES}')
print(f'WORK_DIR:         {WORK_DIR}')
print(f'\nObrazów treningowych: {len(list(IMAGES_SRC.iterdir()))}')

## 2. Instalacja zależności

In [ ]:
!pip install ultralytics -q

## 3. Symlinki obrazów + konwersja COCO → YOLO

In [ ]:
import json
from collections import defaultdict

# --- Symlinki na obrazy treningowe ---
images_dst = WORK_DIR / 'images'
images_dst.mkdir(exist_ok=True)

linked = 0
for img in IMAGES_SRC.iterdir():
    link = images_dst / img.name
    if not link.exists():
        os.symlink(img, link)
        linked += 1
print(f'Symlinki: {linked} nowych, łącznie {len(list(images_dst.iterdir()))}')

# --- Wczytaj anotacje i taksonomię ---
with open(ANNOTATIONS) as f:
    coco = json.load(f)
with open(TAXONOMY) as f:
    taxonomy = json.load(f)

print(f'\nImages:      {len(coco["images"])}')
print(f'Annotations: {len(coco["annotations"])}')
print(f'Categories:  {len(coco["categories"])}')

# --- Mapowanie category_id → indeks YOLO (0-based) ---
cat_ids       = sorted(c['id'] for c in taxonomy['categories'])
cat_id_to_idx = {cid: idx for idx, cid in enumerate(cat_ids)}
names         = {
    idx: next(c['name'] for c in taxonomy['categories'] if c['id'] == cid)
    for idx, cid in enumerate(cat_ids)
}

# --- Konwersja COCO → YOLO labels ---
labels_dir = WORK_DIR / 'labels'
labels_dir.mkdir(exist_ok=True)

img_map     = {img['id']: img for img in coco['images']}
anns_by_img = defaultdict(list)
for ann in coco['annotations']:
    anns_by_img[ann['image_id']].append(ann)

converted = 0
for image_id, anns in anns_by_img.items():
    img_info = img_map[image_id]
    img_w    = img_info['width']
    img_h    = img_info['height']
    stem     = Path(img_info['file_name']).stem

    lines = []
    for ann in anns:
        cat_idx = cat_id_to_idx.get(ann['category_id'])
        if cat_idx is None:
            continue
        x, y, w, h  = ann['bbox']
        x_center    = (x + w / 2) / img_w
        y_center    = (y + h / 2) / img_h
        w_norm      = w / img_w
        h_norm      = h / img_h
        lines.append(f'{cat_idx} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}')

    (labels_dir / f'{stem}.txt').write_text('\n'.join(lines) + '\n')
    converted += 1

print(f'\nSkonwertowano: {converted} plików labels')

## 4. Generuj dataset.yaml

In [ ]:
yaml_lines = [
    f'path: {WORK_DIR}',
    'train: images',
    'val: images',
    f'nc: {len(names)}',
    'names:',
]
for idx, name in sorted(names.items()):
    yaml_lines.append(f'  {idx}: {name}')

yaml_path = WORK_DIR / 'dataset.yaml'
yaml_path.write_text('\n'.join(yaml_lines) + '\n')
print(f'Zapisano: {yaml_path}')
print('\nPierwsze 10 linii:')
print('\n'.join(yaml_lines[:10]))

## 5. Trening modelu

Parametry:
- `yolov8m.pt` — dobry balans jakości i czasu
- `epochs=50` — minimum dla 369 klas
- `imgsz=640` — standard
- `batch=16` — bezpieczne dla T4 15GB

Zmień `_model` na `yolov8l.pt` lub `yolo11m.pt` dla lepszego wyniku.

In [ ]:
from ultralytics import YOLO
import torch

print(f'CUDA dostępne: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

_model  = 'yolov8m.pt'
_epochs = 50
_imgsz  = 640
_batch  = 16
_device = 0

model = YOLO(_model)

results = model.train(
    data=str(yaml_path),
    epochs=_epochs,
    imgsz=_imgsz,
    batch=_batch,
    device=_device,
    project=str(WORK_DIR / 'runs'),
    name='hackology',
    exist_ok=True,
    # augmentacje
    mosaic=1.0,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    degrees=5.0,
)

WEIGHTS_PATH = str(Path(results.save_dir) / 'weights' / 'best.pt')
print(f'\nWagi modelu: {WEIGHTS_PATH}')

## 6. Predykcja na public_test

In [ ]:
# Mapowanie filename → image_id z test_images.json
if TEST_IMAGES_JSON and TEST_IMAGES_JSON.exists():
    with open(TEST_IMAGES_JSON) as f:
        test_data = json.load(f)
    # test_images.json może być listą lub dict z kluczem 'images'
    if isinstance(test_data, list):
        test_images = test_data
    else:
        test_images = test_data.get('images', [])
    filename_to_id = {img['file_name']: img['id'] for img in test_images}
    print(f'Test images: {len(filename_to_id)}')
else:
    print('WARN: test_images.json nie znaleziony')
    filename_to_id = {}

# Mapowanie YOLO idx → category_id
idx_to_cat_id = {idx: cid for cid, idx in cat_id_to_idx.items()}

# Znajdź obrazy testowe
if PUBLIC_TEST_IMAGES.exists():
    test_dir = PUBLIC_TEST_IMAGES
else:
    # fallback — szukaj
    candidates = list(input_root.rglob('public_test'))
    test_dir = candidates[0] / 'images' if candidates else None
    print(f'Znaleziono public_test: {test_dir}')

print(f'Test dir: {test_dir}')
print(f'Test images count: {len(list(test_dir.iterdir())) if test_dir and test_dir.exists() else 0}')

In [ ]:
import sys

model_pred = YOLO(WEIGHTS_PATH)

image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
image_files = sorted(
    p for p in test_dir.iterdir()
    if p.suffix.lower() in image_extensions
)

CONFIDENCE = 0.25
predictions = []
skipped = 0

for img_path in image_files:
    image_id = filename_to_id.get(img_path.name)
    if image_id is None:
        skipped += 1
        continue

    results_pred = model_pred(str(img_path), conf=CONFIDENCE, verbose=False)

    for result in results_pred:
        boxes = result.boxes
        if boxes is None:
            continue
        for i in range(len(boxes)):
            x1, y1, x2, y2 = boxes.xyxy[i].tolist()
            w = x2 - x1
            h = y2 - y1
            cls_id     = int(boxes.cls[i].item())
            category_id = idx_to_cat_id.get(cls_id)
            if category_id is None:
                continue
            score = float(boxes.conf[i].item())
            predictions.append({
                'image_id':    image_id,
                'category_id': category_id,
                'bbox':        [round(x1, 2), round(y1, 2), round(w, 2), round(h, 2)],
                'score':       round(score, 4),
            })

print(f'Predykcji: {len(predictions)}')
print(f'Pominięto (brak image_id): {skipped}')
if predictions:
    print(f'Przykład: {predictions[0]}')

## 7. Zapis predictions.json

In [ ]:
output_path = WORK_DIR / 'predictions.json'
output_path.write_text(json.dumps(predictions, indent=2) + '\n', encoding='utf-8')
print(f'Zapisano {len(predictions)} predykcji do {output_path}')

# Walidacja formatu
required_keys = {'image_id', 'category_id', 'bbox', 'score'}
if predictions:
    missing = required_keys - set(predictions[0].keys())
    if missing:
        print(f'BŁĄD: Brakuje kluczy: {missing}')
    else:
        print('Format OK ✓')
else:
    print('UWAGA: Brak predykcji!')

## 8. Skopiuj wagi do working (do pobrania)

Pobierz `best.pt` z Output i wrzuć do repo lub na HuggingFace.

In [ ]:
import shutil

final_weights = WORK_DIR / 'best.pt'
shutil.copy(WEIGHTS_PATH, final_weights)
print(f'Wagi zapisane: {final_weights}')
print(f'Rozmiar: {final_weights.stat().st_size / 1024 / 1024:.1f} MB')